> 在线讲义：[第 15 章 · 实时与 ros2_control 架构](https://wikieden.github.io/ros2-from-scratch/lessons/15-realtime-ros2-control.html) · 本书《从零学 ROS 2》

# 第 15 章 · 实时与 ros2_control 架构

本章回答：为什么机械臂的关节控制不能直接写成"订阅 → 回调里算 → 发布"的普通节点？"实时"到底要求什么？ros2_control 由哪几块组成，1 kHz 的控制回路在里面怎么跑？

配套讲义：仓库里 `ros2/notes/lessons/15-realtime-ros2-control.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：ros2_control 是 C++ 框架，本机也没有 ROS 2。这里用字典当"接口"，用 mock 硬件（命令原样回写成状态，讲义第 3 步末尾）跑几个周期；最后量一下普通 Python 循环每 1 ms 一拍的实际抖动，看为什么快回路不放 Python。

In [ ]:
import itertools
import statistics
import time

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 2 步 一个周期 | read → update → write | `control_loop(hw, ctrl, 8)` | 0.0, 0.01, …, 0.05, 0.05, 0.05 |
| 第 3 步 命令接口 | 控制器声明要独占的接口 | `ctrl.command_interfaces()` | ['joint1/position'] |
| 第 2 步 接口 = 共享变量 | 读写只是赋值 | `hw.state` / `hw.command` | 两个字典 |
| 第 1 步 实时 = 准时 | 量抖动 | `measure_python_jitter()` | 每次不同；本机约平均 1.5 ms、抖动 0.5 ms |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**类 `MockSystem`**：mock_components/GenericSystem 的样子：write 时把命令记下，下一次 read 时原样当成状态报上来。

In [ ]:
class MockSystem:
    """mock_components/GenericSystem 的样子：write 时把命令记下，下一次 read 时原样当成状态报上来。"""

    def __init__(self, joints: list[str]):
        self.state = {f"{j}/position": 0.0 for j in joints}  # 状态接口：硬件报上来的，只读
        self.command = {f"{j}/position": 0.0 for j in joints}  # 命令接口：发给硬件的
        self._hw = dict(self.state)  # "硬件"里真实的位置

    def read(self):
        self.state.update(self._hw)

    def write(self):
        self._hw.update(self.command)

**类 `StepTowardController`**：一个简单的位置控制器：每拍朝目标走最多 max_step（编的控制律，只为看清 read / update / write 的分工）。

In [ ]:
class StepTowardController:
    """一个简单的位置控制器：每拍朝目标走最多 max_step（编的控制律，只为看清 read / update / write 的分工）。"""

    def __init__(self, joint: str, target: float, max_step: float):
        self.joint, self.target, self.max_step = joint, target, max_step

    def command_interfaces(self) -> list[str]:
        return [f"{self.joint}/position"]

    def update(self, state: dict, command: dict):
        now = state[f"{self.joint}/position"]
        err = self.target - now
        command[f"{self.joint}/position"] = now + max(-self.max_step, min(self.max_step, err))

**函数 `control_loop`**：controller manager 的回路：每个周期 read → update → write。返回每拍读到的关节位置。

In [ ]:
def control_loop(hw: MockSystem, controller: StepTowardController, cycles: int) -> list[float]:
    """controller manager 的回路：每个周期 read → update → write。返回每拍读到的关节位置。"""
    trace = []
    for _ in range(cycles):
        hw.read()  # RM：硬件 → 状态接口
        trace.append(round(hw.state[f"{controller.joint}/position"], 3))
        controller.update(hw.state, hw.command)  # CM：控制器读状态、写命令
        hw.write()  # RM：命令接口 → 硬件
    return trace

**函数 `measure_python_jitter`**：普通 Python：sleep 1 ms 一拍，量实际每拍隔多久。每台电脑、每次结果都不一样。

In [ ]:
def measure_python_jitter(period_s: float = 0.001, n: int = 200) -> dict:
    """普通 Python：sleep 1 ms 一拍，量实际每拍隔多久。每台电脑、每次结果都不一样。"""
    stamps = []
    for _ in range(n):
        stamps.append(time.perf_counter())
        time.sleep(period_s)
    gaps = [(b - a) * 1000 for a, b in itertools.pairwise(stamps)]
    return {"平均 ms": round(statistics.mean(gaps), 3), "最长 ms": round(max(gaps), 3),
            "最大抖动 ms": round(max(abs(g - period_s * 1000) for g in gaps), 3)}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 · 一个周期：read → update → write；mock 硬件把命令原样当成下一拍的状态")
hw = MockSystem(["joint1", "joint2"])
ctrl = StepTowardController("joint1", target=0.05, max_step=0.01)
print("  控制器要独占的命令接口:", ctrl.command_interfaces())
print("  前 8 拍读到的 joint1/position:", control_loop(hw, ctrl, 8))

In [ ]:
print("\n第二步（续）· 接口就是共享变量：读写只是一次赋值")
print("  状态接口:", hw.state)
print("  命令接口:", hw.command)

In [ ]:
print("\n第一步 · 实时要的是准时：量一下普通 Python 每 1 ms 一拍的抖动（每次结果不同）")
print("  ", measure_python_jitter())

## 3. 代码里才看得到的

1. **控制器是编的**：每拍朝目标走最多 0.01，只为看清三步的分工；真位置控制器通常直接转发目标或跑 PID。
2. **mock 硬件有一拍延迟**：write 把命令写进"硬件"，下一次 read 才读回来，所以第 0 拍读到的是 0.0。
3. **Python 循环的抖动**：`time.sleep(0.001)` 在普通 macOS / Linux 上实际要 1.1–1.5 ms，还时不时更久。这就是关节级快回路要放进 C++、实时内核的原因。结果每台电脑都不一样，测试只检查能跑通。

## 4. 自己改一改

- 把 `max_step` 改成 0.02，看几拍到达目标；
- 把 `measure_python_jitter` 的周期改成 0.01（100 Hz），看相对抖动是不是小了。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **实时 = 准时**：周期 + 截止时间 + 小抖动；不分配内存、不缺页、不阻塞、不做 I/O。
- **四件套**：CM 管控制器和节拍，RM 管硬件，控制器算，硬件组件读写。
- **一拍三步**：read 读状态 → update 算命令 → write 写硬件。
- **接口**：状态接口只读，命令接口独占；走共享内存，不走话题。
- **配置**：硬件进 URDF 的 `<ros2_control>`，控制器进 YAML，spawner 拉起来；换 mock 不改控制器。